In [2]:
import os
from pyspark.sql import SparkSession

In [5]:
spark = (
    SparkSession.builder.appName("S3ToChEarthquake")
    .config("spark.jars", "/path/to/postgresql-42.5.0.jar")
    .config("spark.ui.port", "4041")
    .getOrCreate()
)

26/02/13 22:00:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/13 22:00:42 WARN DependencyUtils: Local jar /path/to/postgresql-42.5.0.jar does not exist, skipping.
26/02/13 22:00:42 INFO SparkContext: Running Spark version 4.1.1
26/02/13 22:00:42 INFO SparkContext: OS info Linux, 6.11.0-29-generic, amd64
26/02/13 22:00:42 INFO SparkContext: Java version 17.0.16+8-Debian-1deb12u1
26/02/13 22:00:42 INFO ResourceUtils: ==============================================================
26/02/13 22:00:42 INFO ResourceUtils: No custom resources configured for spark.driver.
26/02/13 22:00:42 INFO ResourceUtils: ==============================================================
26/02/13 22:00:42 INFO SparkContext: Submitted application: S3ToChEarthquake
26/02/13 22:00:42 INFO SecurityManager: Changing view acls to: root
26/02/13 22:00:42

In [4]:
spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

hadoop_conf.set("fs.s3a.threads.keepalivetime", "60") 
hadoop_conf.set("fs.s3a.connection.timeout", "60000")
hadoop_conf.set("fs.s3a.attempts.maximum", "10")
hadoop_conf.set("fs.s3a.connection.establish.timeout", "5000")
hadoop_conf.set("fs.s3a.readahead.range", "65536")

hadoop_conf.set("fs.s3a.multipart.purge.age", "86400")

hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

:: loading settings :: url = jar:file:/home/coder/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9d18e22e-a676-41d4-a1da-66125c226a20;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickho

In [6]:
# ⬇️ Параметры подключения к PostgreSQL
jdbc_url = "jdbc:postgresql://postgres_source:5432/source"
db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")

shops_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shops") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

shop_tz_df = spark.read \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("user", db_user) \
            .option("password", db_password) \
            .option("dbtable", "public.shop_timezone") \
            .option("fetchsize", 1000) \
            .option("driver", "org.postgresql.Driver") \
            .load()

# Регистрируем DataFrame-ы как временные вьюхи
shops_df.createOrReplaceTempView("shops")
shop_tz_df.createOrReplaceTempView("shop_timezone")

In [5]:
shops_df.show()
shop_tz_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [8]:
spark.sql("""
	WITH timezones as (
		SELECT DISTINCT
			plant,
			time_zone
		FROM shop_timezone
		WHERE TRY_CAST(plant AS BIGINT) IS NOT NULL
			AND time_zone != ''
	)

	SELECT
		s.st_id,
		s.shop_name,
		COALESCE(
			TRY_CAST(REGEXP_EXTRACT(t.time_zone, '[1-9]+', 0) AS INT),
			3
		) as tz_code
	FROM shops s
	LEFT JOIN timezones t
	ON s.st_id = t.plant
	ORDER BY st_id
""").show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  846|      Lenta|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  849|   FixPrice|      3|
|  850|     Magnit|      3|
|  851|      Lenta|      3|
+-----+-----------+-------+



In [ ]:
from pyspark.sql.functions import col, regexp_extract, coalesce, lit
from pyspark.sql.types import IntegerType

timezones = (
            shop_tz_df
            .where("""TRY_CAST(plant AS BIGINT) IS NOT NULL AND time_zone != '' """)
            .select("plant", "time_zone")
            .distinct()
            .alias('t')
)

final = (
        shops_df.alias('s')
        .join(timezones, col('s.st_id') == col('t.plant'), 'left')
        .select(
            's.st_id', 
            's.shop_name',
            coalesce(
                regexp_extract(col("t.time_zone"), r'[1-9]+', 0).cast(IntegerType()),
                lit(3)
            ).alias("tz_code")
     
        )
        .orderBy("s.st_id")
)

final.show()

+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  846|      Lenta|      3|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
|  849|   FixPrice|      3|
|  850|     Magnit|      3|
|  851|      Lenta|      3|
+-----+-----------+-------+



In [20]:
spark.stop()